# CornerScout - 05 Modelacion temporal por objetivo

**Responsabilidad:** comparar una referencia liguera previa, un baseline historico
suavizado y un unico candidato regularizado para cada objetivo exportado por `04`.

K-Means se ajusto como descripcion exploratoria en `04`; sus salidas no se evaluan
como modelos ni se usan como variables. No se ejecuta un torneo de algoritmos ni
se introduce una serie de tiempo como modelo separado.

In [ ]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             brier_score_loss, log_loss, mean_absolute_error,
                             mean_poisson_deviance, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize

from analytics.io import data_dir, digest, write_json

DATA = data_dir()
INPUT = DATA / "processed" / "04_features"
OUT = DATA / "processed" / "05_modeling"
OUT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
STAGE_VERSION = "05-modeling-v3-objectives"
SOURCE_CONTRACT = "04-features-v2"
SEED = 42
BOOTSTRAPS = 200
ALPHA_GRID = [2.0, 5.0, 10.0, 20.0]
C_GRID = [0.1, 1.0, 10.0]

PRE_FEATURES = [
    "hist_corners_per_match", "hist_scr15_smoothed", "hist_xg_per_corner",
    "hist_short_share", "hist_high_share", "hist_top_taker_share",
    "hist_dominant_zone_share", "hist_score_losing_share",
    "opp_hist_scr15_conceded_smoothed", "is_home",
]
SCENARIO_NUMERIC = [
    "match_minute", "score_diff", "player_difference", "repeat_corner_60s",
    "seconds_since_previous_same_team_corner", "attacking_players", "defending_players",
]
SCENARIO_CATEGORICAL = ["match_phase", "game_state", "numerical_state", "corner_side"]
PROHIBITED = {"cluster_id", "geometric_name", "height", "end_x", "end_y", "pass_length",
              "execution_type", "delivery_zone", "xg_sequence"}
print({"run_id": RUN_ID, "stage": STAGE_VERSION, "python": platform.python_version(),
       "sklearn": sklearn.__version__})

## 1. Bloque comun: contrato, ventanas y funciones

Las ventanas son de origen movil expansivo y se dimensionan con fechas que
realmente tienen filas `pre_match_ready`. Los primeros ocho partidos de cada
equipo no se imputan ni se reemplazan. Cada ventana aprende imputacion, escalado,
codificacion, regularizacion y fuerza de suavizado solo con su train. No se usa
balanceo automatico. El bootstrap remuestrea partidos completos.

In [ ]:
contract04 = json.loads((INPUT / "contract.json").read_text(encoding="utf-8"))
assert contract04["contract_version"] == SOURCE_CONTRACT
assert contract04["source_contract_version"] == "03-scr15-v2"
assert contract04["counts"]["pre_match_ready"] == 600
assert contract04["clusters_allowed_as_model_features"] is False

def export_meta(name):
    return next(item for item in contract04["exports"] if item["file"] == name)

def load_table(name):
    path = INPUT / name
    assert digest(path) == export_meta(name)["sha256"]
    frame = pd.read_parquet(path)
    if "match_date" in frame:
        frame["match_date"] = pd.to_datetime(frame.match_date).dt.normalize()
    return frame

scr_pre = load_table("model_scr15_pre_match.parquet")
scr_scenario = load_table("model_scr15_scenario.parquet")
short_data = load_table("model_short_direct.parquet")
zone_data = load_table("model_delivery_zone.parquet")
count_data = load_table("model_corner_count.parquet")
observed = load_table("team_match_observed.parquet")
features = load_table("pre_match_features.parquet")
corners = load_table("corners_engineered.parquet")

assert len(scr_pre) == len(scr_scenario) == 3039
assert len(short_data) == 3838 and len(zone_data) == 3374
assert len(count_data) == 760 and int(count_data.pre_match_ready.sum()) == 600
assert int(count_data.n_corners.eq(0).sum()) == 17
for frame in [scr_pre, scr_scenario, short_data, zone_data, count_data]:
    assert PROHIBITED.isdisjoint(set(frame.columns) - {"delivery_zone"})

ready_dates = np.array(sorted(count_data.loc[count_data.pre_match_ready, "match_date"].unique()))
assert len(ready_dates) >= 20
initial_n = max(8, int(np.floor(len(ready_dates) * 0.40)))
final_n = max(4, int(np.ceil(len(ready_dates) * 0.15)))
development_dates = ready_dates[initial_n:-final_n]
development_blocks = [block for block in np.array_split(development_dates, 3) if len(block)]
assert len(development_blocks) == 3
WINDOWS = []
for index, block in enumerate(development_blocks, start=1):
    WINDOWS.append({"name": f"development_{index}", "start": pd.Timestamp(block[0]),
                    "end": pd.Timestamp(block[-1]) + pd.Timedelta(days=1), "role": "selection"})
FINAL_START = pd.Timestamp(ready_dates[-final_n])
FINAL_END = pd.Timestamp(ready_dates[-1]) + pd.Timedelta(days=1)
assert WINDOWS[-1]["end"] <= FINAL_START

window_table = pd.DataFrame(WINDOWS + [{"name": "final", "start": FINAL_START,
                                        "end": FINAL_END, "role": "confirmation_only"}])
window_table["train_rows_team_match"] = window_table.start.map(
    lambda start: int(((count_data.pre_match_ready) & (count_data.match_date < start)).sum())
)
window_table["test_rows_team_match"] = window_table.apply(
    lambda row: int((count_data.pre_match_ready & count_data.match_date.ge(row.start)
                     & count_data.match_date.lt(row.end)).sum()), axis=1
)
assert window_table.train_rows_team_match.gt(0).all() and window_table.test_rows_team_match.gt(0).all()
display(window_table)

def inner_split(train):
    dates = np.array(sorted(train.match_date.unique()))
    cut = pd.Timestamp(dates[max(1, int(len(dates) * 0.8))])
    fit = train[train.match_date < cut]
    validation = train[train.match_date >= cut]
    assert len(fit) and len(validation) and fit.match_date.max() < validation.match_date.min()
    return fit, validation

def make_classifier(numeric, categorical, c=1.0):
    transformers = []
    if numeric:
        transformers.append(("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())
        ]), numeric))
    if categorical:
        transformers.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical))
    return Pipeline([
        ("preprocess", ColumnTransformer(transformers)),
        ("model", LogisticRegression(C=c, max_iter=3000, solver="lbfgs", random_state=SEED)),
    ])

def tune_classifier(train, target, feature_sets):
    fit, validation = inner_split(train)
    rows = []
    for set_name, numeric, categorical in feature_sets:
        for c in C_GRID:
            model = make_classifier(numeric, categorical, c)
            model.fit(fit[numeric + categorical], fit[target])
            probability = model.predict_proba(validation[numeric + categorical])
            score = log_loss(validation[target], probability, labels=model.classes_)
            rows.append({"feature_set": set_name, "C": c, "inner_log_loss": score})
    chosen = min(rows, key=lambda row: (row["inner_log_loss"], row["feature_set"], row["C"]))
    return chosen, pd.DataFrame(rows)

def tune_binary_smoothing(train, target, raw_column, exposure_column):
    fit, validation = inner_split(train)
    prior = float(fit[target].mean())
    rows = []
    for alpha in ALPHA_GRID:
        probability = (validation[raw_column] * validation[exposure_column] + alpha * prior) / (validation[exposure_column] + alpha)
        rows.append({"alpha": alpha, "inner_log_loss": log_loss(validation[target], probability.clip(1e-6, 1-1e-6))})
    return min(rows, key=lambda row: (row["inner_log_loss"], row["alpha"])), pd.DataFrame(rows)

def binary_metrics(y, probability):
    probability = np.clip(np.asarray(probability, dtype=float), 1e-6, 1-1e-6)
    y = np.asarray(y, dtype=int)
    return {"n": len(y), "positive": int(y.sum()), "prevalence": float(y.mean()),
            "average_precision": average_precision_score(y, probability),
            "brier": brier_score_loss(y, probability),
            "log_loss": log_loss(y, probability),
            "calibration_gap": float(abs(probability.mean() - y.mean())),
            "roc_auc": roc_auc_score(y, probability)}

def bootstrap_binary(frame, target, candidate, reference, window, objective):
    rng = np.random.default_rng(SEED)
    match_ids = frame.match_id.unique()
    rows = []
    for iteration in range(BOOTSTRAPS):
        sampled = rng.choice(match_ids, len(match_ids), replace=True)
        indices = np.concatenate([frame.index[frame.match_id.eq(match_id)].to_numpy() for match_id in sampled])
        sample = frame.loc[indices]
        rows.append({"objective": objective, "window": window, "iteration": iteration,
                     "brier_delta": brier_score_loss(sample[target], sample[candidate]) - brier_score_loss(sample[target], sample[reference]),
                     "ap_delta": average_precision_score(sample[target], sample[candidate]) - average_precision_score(sample[target], sample[reference])})
    return rows

def choose_binary(metrics, objective):
    dev = metrics[(metrics.objective == objective) & metrics.role.eq("selection")]
    rows = []
    for window in dev.window.unique():
        block = dev[dev.window.eq(window)].set_index("model")
        reference_name = block.loc[[name for name in ["league_reference", "historical_baseline"] if name in block.index], "log_loss"].idxmin()
        candidate = block.loc["candidate"]
        reference = block.loc[reference_name]
        passed = (candidate.brier < reference.brier and candidate.log_loss < reference.log_loss
                  and candidate.average_precision > reference.average_precision
                  and candidate.calibration_gap <= reference.calibration_gap + 0.01)
        rows.append({"objective": objective, "window": window, "reference": reference_name,
                     "quality_better": bool(candidate.brier < reference.brier and candidate.log_loss < reference.log_loss),
                     "discrimination_better": bool(candidate.average_precision > reference.average_precision),
                     "calibration_not_degraded": bool(candidate.calibration_gap <= reference.calibration_gap + 0.01),
                     "passed": bool(passed)})
    checks = pd.DataFrame(rows)
    wins = int(checks.passed.sum())
    best_reference = dev[dev.model.isin(["league_reference", "historical_baseline"])].groupby("model").log_loss.mean().idxmin()
    winner = "candidate" if wins >= 2 else best_reference
    return winner, checks

## 2. Persistencia de habitos

Antes de modelar corto/directo y zona se comprueba si los ocho partidos previos
contienen informacion sobre el siguiente equipo-partido. Esto justifica estudiar
el objetivo, no garantiza que el candidato sea promovido.

In [ ]:
habit = observed.merge(features, on=["match_id", "match_date", "team", "opponent", "is_home"],
                       how="left", validate="one_to_one", suffixes=("", "_history"))
habit = habit[habit.pre_match_ready & habit.n_corners.gt(0)].copy()
habit["current_short_share"] = habit.n_short / habit.n_corners
short_rho, short_p = spearmanr(habit.hist_short_share, habit.current_short_share)

zone_classes = sorted(zone_data.delivery_zone.unique())
zone_corner_lookup = corners[corners.direct_delivery_valid].copy()
zone_rows = []
for row in features[features.pre_match_ready].itertuples():
    history_ids = json.loads(row.history_match_ids)
    historical = zone_corner_lookup[zone_corner_lookup.match_id.isin(history_ids) & zone_corner_lookup.team.eq(row.team)]
    current = zone_corner_lookup[zone_corner_lookup.match_id.eq(row.match_id) & zone_corner_lookup.team.eq(row.team)]
    if len(current) == 0:
        continue
    hist_counts = historical.delivery_zone.value_counts()
    current_counts = current.delivery_zone.value_counts()
    hist_top = hist_counts.idxmax() if len(hist_counts) else None
    current_top = current_counts.idxmax()
    zone_rows.append({"match_id": row.match_id, "team": row.team, "historical_top_zone": hist_top,
                      "current_top_zone": current_top, "same_top_zone": hist_top == current_top,
                      **{f"hist_zone_count_{zone}": int(hist_counts.get(zone, 0)) for zone in zone_classes}})
zone_persistence_rows = pd.DataFrame(zone_rows)
zone_top_accuracy = float(zone_persistence_rows.same_top_zone.mean())
zone_chance = float(zone_persistence_rows.current_top_zone.value_counts(normalize=True).max())

persistence = pd.DataFrame([
    {"habit": "short_share", "n_team_matches": len(habit), "statistic": "spearman_rho",
     "value": float(short_rho), "comparison": float(short_p),
     "interpretation": "historical share versus next team-match share"},
    {"habit": "dominant_delivery_zone", "n_team_matches": len(zone_persistence_rows),
     "statistic": "top_zone_accuracy", "value": zone_top_accuracy, "comparison": zone_chance,
     "interpretation": "historical dominant zone versus next; comparison is majority-class chance"},
])
display(persistence.round(4))

## 3. Objetivo SCR-15 binario

In [ ]:
history_raw = features[["match_id", "team", "hist_scr15_raw"]]
scr = scr_scenario.merge(history_raw, on=["match_id", "team"], how="left", validate="many_to_one")
scr["history_exposure"] = (scr.hist_corners_per_match * 8).round().clip(lower=1)
scr_feature_sets = [
    ("pre_match", PRE_FEATURES, []),
    ("pre_match_plus_scenario", PRE_FEATURES + SCENARIO_NUMERIC, SCENARIO_CATEGORICAL),
]
scr_metric_rows, scr_predictions, tuning_rows, bootstrap_rows, fitted_scr = [], [], [], [], {}
for window in WINDOWS + [{"name": "final", "start": FINAL_START, "end": FINAL_END, "role": "confirmation_only"}]:
    train = scr[scr.match_date < window["start"]].copy()
    test = scr[scr.match_date.ge(window["start"]) & scr.match_date.lt(window["end"])].copy()
    assert set(train.match_id).isdisjoint(test.match_id) and train.match_date.max() < test.match_date.min()
    smooth, smooth_grid = tune_binary_smoothing(train, "shot_within_15s", "hist_scr15_raw", "history_exposure")
    chosen, candidate_grid = tune_classifier(train, "shot_within_15s", scr_feature_sets)
    numeric, categorical = next((n, c) for name, n, c in scr_feature_sets if name == chosen["feature_set"])
    model = make_classifier(numeric, categorical, chosen["C"])
    model.fit(train[numeric + categorical], train.shot_within_15s)
    fitted_scr[window["name"]] = (model, numeric + categorical, chosen, smooth)
    league = float(train.shot_within_15s.mean())
    probability = {
        "league_reference": np.full(len(test), league),
        "historical_baseline": ((test.hist_scr15_raw * test.history_exposure + smooth["alpha"] * league)
                                / (test.history_exposure + smooth["alpha"])).to_numpy(),
        "candidate": model.predict_proba(test[numeric + categorical])[:, 1],
    }
    pred = test[["event_id", "match_id", "match_date", "team", "shot_within_15s"]].copy()
    pred["window"] = window["name"]
    for name, values in probability.items():
        pred[f"p_{name}"] = values
        scr_metric_rows.append({"objective": "scr15", "window": window["name"], "role": window["role"],
                                "model": name, **binary_metrics(test.shot_within_15s, values)})
    if window["role"] == "selection":
        bootstrap_rows.extend(bootstrap_binary(pred, "shot_within_15s", "p_candidate", "p_historical_baseline", window["name"], "scr15"))
    tuning_rows.extend(candidate_grid.assign(objective="scr15", window=window["name"], parameter="C").to_dict("records"))
    tuning_rows.extend(smooth_grid.assign(objective="scr15", window=window["name"], parameter="smoothing_alpha").to_dict("records"))
    scr_predictions.append(pred)
scr_metrics = pd.DataFrame(scr_metric_rows)
scr_predictions = pd.concat(scr_predictions, ignore_index=True)
scr_winner, scr_checks = choose_binary(scr_metrics, "scr15")
display(scr_metrics.round(4))
display(scr_checks)
print("Ganador SCR-15:", scr_winner)

## 4. Objetivo corto/directo binario

El target es el proxy versionado. Los predictores terminan antes del cobro: no
incluyen altura, tecnica, longitud, receptor, destino ni otra variable observada
durante la ejecucion.

In [ ]:
short = short_data[short_data.pre_match_ready].copy()
short["history_exposure"] = (short.hist_corners_per_match * 8).round().clip(lower=1)
short_features = [("pre_kick", PRE_FEATURES + SCENARIO_NUMERIC, SCENARIO_CATEGORICAL)]
assert PROHIBITED.isdisjoint(set(PRE_FEATURES + SCENARIO_NUMERIC + SCENARIO_CATEGORICAL))
short_metric_rows, short_predictions, fitted_short = [], [], {}
for window in WINDOWS + [{"name": "final", "start": FINAL_START, "end": FINAL_END, "role": "confirmation_only"}]:
    train = short[short.match_date < window["start"]].copy()
    test = short[short.match_date.ge(window["start"]) & short.match_date.lt(window["end"])].copy()
    smooth, smooth_grid = tune_binary_smoothing(train, "short_proxy", "hist_short_share", "history_exposure")
    chosen, candidate_grid = tune_classifier(train, "short_proxy", short_features)
    numeric, categorical = short_features[0][1], short_features[0][2]
    model = make_classifier(numeric, categorical, chosen["C"])
    model.fit(train[numeric + categorical], train.short_proxy)
    fitted_short[window["name"]] = (model, numeric + categorical, chosen, smooth)
    league = float(train.short_proxy.mean())
    probability = {
        "league_reference": np.full(len(test), league),
        "historical_baseline": ((test.hist_short_share * test.history_exposure + smooth["alpha"] * league)
                                / (test.history_exposure + smooth["alpha"])).to_numpy(),
        "candidate": model.predict_proba(test[numeric + categorical])[:, 1],
    }
    pred = test[["event_id", "match_id", "match_date", "team", "short_proxy"]].copy()
    pred["window"] = window["name"]
    for name, values in probability.items():
        pred[f"p_{name}"] = values
        short_metric_rows.append({"objective": "short_direct", "window": window["name"], "role": window["role"],
                                  "model": name, **binary_metrics(test.short_proxy, values)})
    if window["role"] == "selection":
        bootstrap_rows.extend(bootstrap_binary(pred, "short_proxy", "p_candidate", "p_historical_baseline", window["name"], "short_direct"))
    tuning_rows.extend(candidate_grid.assign(objective="short_direct", window=window["name"], parameter="C").to_dict("records"))
    tuning_rows.extend(smooth_grid.assign(objective="short_direct", window=window["name"], parameter="smoothing_alpha").to_dict("records"))
    short_predictions.append(pred)
short_metrics = pd.DataFrame(short_metric_rows)
short_predictions = pd.concat(short_predictions, ignore_index=True)
short_winner, short_checks = choose_binary(short_metrics, "short_direct")
display(short_metrics.round(4))
display(short_checks)
print("Ganador corto/directo:", short_winner)

## 5. Objetivo zona multiclase opcional

In [ ]:
zone = zone_data[zone_data.pre_match_ready].copy()
zone = zone.merge(features[["match_id", "team", "history_match_ids"]], on=["match_id", "team"],
                  how="left", validate="many_to_one")
for zone_name in zone_classes:
    lookup = zone_persistence_rows.set_index(["match_id", "team"])[f"hist_zone_count_{zone_name}"]
    zone[f"hist_zone_count_{zone_name}"] = [lookup.get((row.match_id, row.team), 0) for row in zone.itertuples()]
support_total = zone.delivery_zone.value_counts().reindex(zone_classes, fill_value=0)
support_teams = zone.groupby("delivery_zone").team.nunique().reindex(zone_classes, fill_value=0)
zone_gate = bool(support_total.min() >= 100 and support_teams.min() >= 10 and zone_top_accuracy > zone_chance)
zone_support = pd.DataFrame({"delivery_zone": zone_classes, "n": support_total.values,
                             "teams": support_teams.values})

def multiclass_metrics(y, probability, classes):
    encoded = label_binarize(y, classes=classes)
    return {"n": len(y), "log_loss": log_loss(y, probability, labels=classes),
            "multiclass_brier": float(np.mean(np.sum((encoded - probability) ** 2, axis=1))),
            "macro_average_precision": float(np.mean([average_precision_score(encoded[:, i], probability[:, i]) for i in range(len(classes))])),
            "calibration_gap": float(np.mean(np.abs(probability.mean(axis=0) - encoded.mean(axis=0))))}

zone_metric_rows, zone_predictions, fitted_zone = [], [], {}
if zone_gate:
    zone_features = [("pre_kick", PRE_FEATURES + SCENARIO_NUMERIC, SCENARIO_CATEGORICAL)]
    for window in WINDOWS + [{"name": "final", "start": FINAL_START, "end": FINAL_END, "role": "confirmation_only"}]:
        train = zone[zone.match_date < window["start"]].copy()
        test = zone[zone.match_date.ge(window["start"]) & zone.match_date.lt(window["end"])].copy()
        fit, validation = inner_split(train)
        prior_fit = fit.delivery_zone.value_counts().reindex(zone_classes, fill_value=0).to_numpy(dtype=float)
        alpha_scores = []
        for alpha in ALPHA_GRID:
            hist = validation[[f"hist_zone_count_{name}" for name in zone_classes]].to_numpy(dtype=float)
            probability = (hist + alpha * prior_fit / prior_fit.sum())
            probability /= probability.sum(axis=1, keepdims=True)
            alpha_scores.append({"alpha": alpha, "inner_log_loss": log_loss(validation.delivery_zone, probability, labels=zone_classes)})
        smooth = min(alpha_scores, key=lambda row: (row["inner_log_loss"], row["alpha"]))
        chosen, candidate_grid = tune_classifier(train, "delivery_zone", zone_features)
        numeric, categorical = zone_features[0][1], zone_features[0][2]
        model = make_classifier(numeric, categorical, chosen["C"])
        model.fit(train[numeric + categorical], train.delivery_zone)
        fitted_zone[window["name"]] = (model, numeric + categorical, chosen, smooth)
        prior = train.delivery_zone.value_counts().reindex(zone_classes, fill_value=0).to_numpy(dtype=float)
        prior /= prior.sum()
        hist = test[[f"hist_zone_count_{name}" for name in zone_classes]].to_numpy(dtype=float)
        baseline = hist + smooth["alpha"] * prior
        baseline /= baseline.sum(axis=1, keepdims=True)
        candidate_raw = model.predict_proba(test[numeric + categorical])
        candidate = np.column_stack([candidate_raw[:, list(model.classes_).index(name)] for name in zone_classes])
        probabilities = {"league_reference": np.tile(prior, (len(test), 1)),
                         "historical_baseline": baseline, "candidate": candidate}
        pred = test[["event_id", "match_id", "match_date", "team", "delivery_zone"]].copy()
        pred["window"] = window["name"]
        for model_name, values in probabilities.items():
            for i, class_name in enumerate(zone_classes):
                pred[f"p_{model_name}_{class_name}"] = values[:, i]
            zone_metric_rows.append({"objective": "delivery_zone", "window": window["name"],
                                     "role": window["role"], "model": model_name,
                                     **multiclass_metrics(test.delivery_zone, values, zone_classes)})
        zone_predictions.append(pred)
        tuning_rows.extend(candidate_grid.assign(objective="delivery_zone", window=window["name"], parameter="C").to_dict("records"))
    zone_metrics = pd.DataFrame(zone_metric_rows)
    zone_predictions = pd.concat(zone_predictions, ignore_index=True)
    zone_checks = []
    for window in [item["name"] for item in WINDOWS]:
        block = zone_metrics[zone_metrics.window.eq(window)].set_index("model")
        reference_name = block.loc[["league_reference", "historical_baseline"], "log_loss"].idxmin()
        candidate, reference = block.loc["candidate"], block.loc[reference_name]
        passed = (candidate.log_loss < reference.log_loss and candidate.multiclass_brier < reference.multiclass_brier
                  and candidate.macro_average_precision > reference.macro_average_precision
                  and candidate.calibration_gap <= reference.calibration_gap + 0.01)
        zone_checks.append({"objective": "delivery_zone", "window": window, "reference": reference_name, "passed": bool(passed)})
    zone_checks = pd.DataFrame(zone_checks)
    zone_winner = "candidate" if zone_checks.passed.sum() >= 2 else zone_metrics[zone_metrics.role.eq("selection") & zone_metrics.model.ne("candidate")].groupby("model").log_loss.mean().idxmin()
else:
    zone_metrics = pd.DataFrame()
    zone_predictions = pd.DataFrame()
    zone_checks = pd.DataFrame([{"objective": "delivery_zone", "passed": False,
                                 "reason": "support or persistence gate failed"}])
    zone_winner = "not_modeled"
display(zone_support)
print({"zone_gate": zone_gate, "persistence": zone_top_accuracy, "chance": zone_chance,
       "winner": zone_winner})

## 6. Objetivo conteo por equipo-partido opcional

In [ ]:
count = count_data[count_data.pre_match_ready].copy()
count_features = ["hist_corners_per_match", "opp_hist_scr15_conceded_smoothed", "is_home"]
count_predictions, count_metric_rows, fitted_count = [], [], {}
initial_train = count[count.match_date < WINDOWS[0]["start"]]
initial_mean = initial_train.n_corners.mean()
raw_dispersion = float(initial_train.n_corners.var(ddof=1) / initial_mean)
dispersion_probe = Pipeline([("impute", SimpleImputer(strategy="median")),
                             ("scale", StandardScaler()),
                             ("model", PoissonRegressor(alpha=0.1, max_iter=1000))])
dispersion_probe.fit(initial_train[count_features], initial_train.n_corners)
probe_mean = np.clip(dispersion_probe.predict(initial_train[count_features]), 1e-6, None)
conditional_dispersion = float(np.sum((initial_train.n_corners.to_numpy() - probe_mean) ** 2 / probe_mean)
                               / max(1, len(initial_train) - len(count_features) - 1))
count_family = "poisson" if conditional_dispersion <= 1.5 else "negative_binomial_required"
count_gate = count_family == "poisson"
if count_gate:
    for window in WINDOWS + [{"name": "final", "start": FINAL_START, "end": FINAL_END, "role": "confirmation_only"}]:
        train = count[count.match_date < window["start"]].copy()
        test = count[count.match_date.ge(window["start"]) & count.match_date.lt(window["end"])].copy()
        fit, validation = inner_split(train)
        count_smoothing = []
        for alpha in ALPHA_GRID:
            prediction = (validation.hist_corners_per_match * 8 + alpha * fit.n_corners.mean()) / (8 + alpha)
            count_smoothing.append({"alpha": alpha, "inner_mae": mean_absolute_error(validation.n_corners, prediction)})
        smooth = min(count_smoothing, key=lambda row: (row["inner_mae"], row["alpha"]))
        poisson_grid = []
        for alpha in [0.01, 0.1, 1.0]:
            model = Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler()),
                              ("model", PoissonRegressor(alpha=alpha, max_iter=1000))])
            model.fit(fit[count_features], fit.n_corners)
            prediction = np.clip(model.predict(validation[count_features]), 1e-6, None)
            poisson_grid.append({"alpha": alpha, "inner_deviance": mean_poisson_deviance(validation.n_corners, prediction)})
        chosen = min(poisson_grid, key=lambda row: (row["inner_deviance"], row["alpha"]))
        model = Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler()),
                          ("model", PoissonRegressor(alpha=chosen["alpha"], max_iter=1000))])
        model.fit(train[count_features], train.n_corners)
        fitted_count[window["name"]] = (model, count_features, chosen, smooth)
        league = float(train.n_corners.mean())
        predictions = {"league_reference": np.full(len(test), league),
                       "historical_baseline": ((test.hist_corners_per_match * 8 + smooth["alpha"] * league) / (8 + smooth["alpha"])).to_numpy(),
                       "candidate": np.clip(model.predict(test[count_features]), 1e-6, None)}
        pred = test[["match_id", "match_date", "team", "n_corners"]].copy()
        pred["window"] = window["name"]
        for model_name, values in predictions.items():
            pred[f"prediction_{model_name}"] = values
            count_metric_rows.append({"objective": "corner_count", "window": window["name"],
                                      "role": window["role"], "model": model_name, "n": len(test),
                                      "mae": mean_absolute_error(test.n_corners, values),
                                      "poisson_deviance": mean_poisson_deviance(test.n_corners, np.clip(values, 1e-6, None))})
        if window["role"] == "selection":
            rng = np.random.default_rng(SEED)
            match_ids = pred.match_id.unique()
            for iteration in range(BOOTSTRAPS):
                sampled = rng.choice(match_ids, len(match_ids), replace=True)
                indices = np.concatenate([pred.index[pred.match_id.eq(match_id)].to_numpy() for match_id in sampled])
                sample = pred.loc[indices]
                bootstrap_rows.append({"objective": "corner_count", "window": window["name"],
                                       "iteration": iteration,
                                       "mae_delta": mean_absolute_error(sample.n_corners, sample.prediction_candidate) - mean_absolute_error(sample.n_corners, sample.prediction_historical_baseline),
                                       "deviance_delta": mean_poisson_deviance(sample.n_corners, sample.prediction_candidate) - mean_poisson_deviance(sample.n_corners, sample.prediction_historical_baseline)})
        count_predictions.append(pred)
        tuning_rows.extend(pd.DataFrame(poisson_grid).assign(objective="corner_count", window=window["name"], parameter="poisson_alpha").to_dict("records"))
    count_metrics = pd.DataFrame(count_metric_rows)
    count_predictions = pd.concat(count_predictions, ignore_index=True)
    count_checks = []
    for window in [item["name"] for item in WINDOWS]:
        block = count_metrics[count_metrics.window.eq(window)].set_index("model")
        reference_name = block.loc[["league_reference", "historical_baseline"], "poisson_deviance"].idxmin()
        passed = (block.loc["candidate", "poisson_deviance"] < block.loc[reference_name, "poisson_deviance"]
                  and block.loc["candidate", "mae"] < block.loc[reference_name, "mae"])
        count_checks.append({"objective": "corner_count", "window": window, "reference": reference_name, "passed": bool(passed)})
    count_checks = pd.DataFrame(count_checks)
    count_winner = "candidate" if count_checks.passed.sum() >= 2 else count_metrics[count_metrics.role.eq("selection") & count_metrics.model.ne("candidate")].groupby("model").poisson_deviance.mean().idxmin()
else:
    count_metrics = pd.DataFrame()
    count_predictions = pd.DataFrame()
    count_checks = pd.DataFrame([{"objective": "corner_count", "passed": False,
                                  "reason": "dispersion requires a negative-binomial implementation"}])
    count_winner = "not_modeled"
print({"raw_dispersion": raw_dispersion, "conditional_dispersion": conditional_dispersion,
       "family": count_family, "winner": count_winner})

## 7. Seleccion, artefactos y model card

La regla binaria exige mejor Brier y log loss, mejor AP y calibracion no degradada
en al menos dos de tres ventanas de desarrollo. La ventana final nunca cambia la
decision. No se reporta un corte clasificatorio. El artefacto retrospectivo se
ajusta solo con fechas anteriores al tramo final; el artefacto `refit_full` usa
toda la temporada y no se presenta como reevaluado.

In [ ]:
all_metrics = pd.concat([frame for frame in [scr_metrics, short_metrics, zone_metrics, count_metrics] if len(frame)], ignore_index=True)
all_checks = pd.concat([scr_checks, short_checks, zone_checks, count_checks], ignore_index=True, sort=False)
bootstrap = pd.DataFrame(bootstrap_rows)
tuning = pd.DataFrame(tuning_rows)

winner_rows = [
    {"objective": "scr15", "winner": scr_winner, "modeled": True,
     "justification": f"candidate passed {int(scr_checks.passed.sum())}/3 development windows"},
    {"objective": "short_direct", "winner": short_winner, "modeled": True,
     "justification": f"candidate passed {int(short_checks.passed.sum())}/3 development windows; persistence rho={short_rho:.3f}"},
    {"objective": "delivery_zone", "winner": zone_winner, "modeled": zone_gate,
     "justification": (f"support min={int(zone_support.n.min())}, teams min={int(zone_support.teams.min())}, persistence={zone_top_accuracy:.3f} vs {zone_chance:.3f}")},
    {"objective": "corner_count", "winner": count_winner, "modeled": count_gate,
     "justification": f"variance/mean={raw_dispersion:.3f}; conditional dispersion={conditional_dispersion:.3f}; family={count_family}"},
]
winners = pd.DataFrame(winner_rows)
display(winners)

# Export candidate pipelines even when the reference wins; promotion status lives in winners.
artifact_records = []
def export_candidate(objective, fitted, data, target, feature_sets):
    retrospective_model, retrospective_columns, _, _ = fitted["final"]
    retrospective_path = OUT / f"{objective}_candidate_retrospective.joblib"
    joblib.dump(retrospective_model, retrospective_path)
    full_choice, _ = tune_classifier(data, target, feature_sets)
    numeric, categorical = next((n, c) for name, n, c in feature_sets if name == full_choice["feature_set"])
    full_model = make_classifier(numeric, categorical, full_choice["C"])
    full_model.fit(data[numeric + categorical], data[target])
    full_path = OUT / f"{objective}_candidate_refit_full.joblib"
    joblib.dump(full_model, full_path)
    schema = {"objective": objective, "target": target, "input_columns": numeric + categorical,
              "retrospective_trained_before": str(FINAL_START.date()),
              "refit_full_evaluation_status": "not retrospectively evaluated",
              "winner": winners.set_index("objective").loc[objective, "winner"]}
    schema_path = OUT / f"{objective}_input_schema.json"
    write_json(schema_path, schema)
    for path, role in [(retrospective_path, "retrospective_candidate"), (full_path, "refit_full_candidate"), (schema_path, "schema")]:
        artifact_records.append({"file": path.name, "role": role, "sha256": digest(path), "bytes": path.stat().st_size})

export_candidate("scr15", fitted_scr, scr, "shot_within_15s", scr_feature_sets)
export_candidate("short_direct", fitted_short, short, "short_proxy", short_features)
if zone_gate:
    export_candidate("delivery_zone", fitted_zone, zone, "delivery_zone", zone_features)
if count_gate:
    retrospective_model = fitted_count["final"][0]
    path = OUT / "corner_count_candidate_retrospective.joblib"
    joblib.dump(retrospective_model, path)
    full_model = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                           ("model", PoissonRegressor(alpha=fitted_count["final"][2]["alpha"], max_iter=1000))])
    full_model.fit(count[count_features], count.n_corners)
    full_path = OUT / "corner_count_candidate_refit_full.joblib"
    joblib.dump(full_model, full_path)
    schema_path = OUT / "corner_count_input_schema.json"
    write_json(schema_path, {"objective": "corner_count", "target": "n_corners",
                             "input_columns": count_features, "family": count_family,
                             "refit_full_evaluation_status": "not retrospectively evaluated",
                             "winner": count_winner})
    for item, role in [(path, "retrospective_candidate"), (full_path, "refit_full_candidate"), (schema_path, "schema")]:
        artifact_records.append({"file": item.name, "role": role, "sha256": digest(item), "bytes": item.stat().st_size})

# Artefactos seleccionados: separados entre reevaluacion retrospectiva y reajuste total.
if scr_winner == "candidate":
    selected_scr_retro = fitted_scr["final"][0]
    selected_scr_full = joblib.load(OUT / "scr15_candidate_refit_full.joblib")
else:
    selected_scr_retro = Pipeline([("model", DummyClassifier(strategy="prior"))]).fit(
        scr.loc[scr.match_date < FINAL_START, ["is_home"]], scr.loc[scr.match_date < FINAL_START, "shot_within_15s"]
    )
    selected_scr_full = Pipeline([("model", DummyClassifier(strategy="prior"))]).fit(scr[["is_home"]], scr.shot_within_15s)
for model, suffix in [(selected_scr_retro, "retrospective"), (selected_scr_full, "refit_full")]:
    path = OUT / f"scr15_selected_{suffix}.joblib"
    joblib.dump(model, path)
    artifact_records.append({"file": path.name, "role": f"selected_{suffix}", "sha256": digest(path), "bytes": path.stat().st_size})
selected_scr_schema = OUT / "scr15_selected_input_schema.json"
write_json(selected_scr_schema, {"objective": "scr15", "winner": scr_winner,
                                 "input_columns": (["is_home"] if scr_winner == "league_reference" else fitted_scr["final"][1]),
                                 "pipeline": "preprocessing plus selected model; league reference needs no transformation",
                                 "retrospective_trained_before": str(FINAL_START.date()),
                                 "refit_full_evaluation_status": "not retrospectively evaluated"})
artifact_records.append({"file": selected_scr_schema.name, "role": "selected_schema",
                         "sha256": digest(selected_scr_schema), "bytes": selected_scr_schema.stat().st_size})

if short_winner == "candidate":
    for source, suffix in [(OUT / "short_direct_candidate_retrospective.joblib", "retrospective"),
                           (OUT / "short_direct_candidate_refit_full.joblib", "refit_full")]:
        path = OUT / f"short_direct_selected_{suffix}.joblib"
        joblib.dump(joblib.load(source), path)
        artifact_records.append({"file": path.name, "role": f"selected_{suffix}", "sha256": digest(path), "bytes": path.stat().st_size})
if count_winner == "candidate":
    for source, suffix in [(OUT / "corner_count_candidate_retrospective.joblib", "retrospective"),
                           (OUT / "corner_count_candidate_refit_full.joblib", "refit_full")]:
        path = OUT / f"corner_count_selected_{suffix}.joblib"
        joblib.dump(joblib.load(source), path)
        artifact_records.append({"file": path.name, "role": f"selected_{suffix}", "sha256": digest(path), "bytes": path.stat().st_size})

tables = {"temporal_windows.parquet": window_table, "habit_persistence.parquet": persistence,
          "temporal_metrics.parquet": all_metrics, "selection_checks.parquet": all_checks,
          "bootstrap_by_match.parquet": bootstrap, "tuning_development.parquet": tuning,
          "objective_winners.parquet": winners, "zone_support.parquet": zone_support,
          "scr15_predictions.parquet": scr_predictions, "short_predictions.parquet": short_predictions}
if len(zone_predictions): tables["zone_predictions.parquet"] = zone_predictions
if len(count_predictions): tables["count_predictions.parquet"] = count_predictions
for filename, frame in tables.items():
    path = OUT / filename
    frame.to_parquet(path, index=False)
    artifact_records.append({"file": filename, "role": "evidence", "rows": len(frame),
                             "columns": len(frame.columns), "sha256": digest(path)})

limitations = [
    "Single historical season; no external-season validation.",
    "The final period did not influence model decisions, but 04 quality checks inspected full-season aggregate counts, so it is not a pristine untouched holdout.",
    "Short/direct target is a calibrated proxy based on 40 legacy labels, not independently validated ground truth.",
    "Scenario model applies only once a corner is awarded; it is not a pure pre-match estimate.",
    "K-Means outputs are excluded from every model input.",
    "Refit-full artifacts have not been retrospectively evaluated.",
]
winner_markdown = chr(10).join(
    ["| objective | winner | modeled | justification |", "|---|---|---:|---|"]
    + [f"| {row.objective} | {row.winner} | {row.modeled} | {row.justification} |" for row in winners.itertuples()]
)
metric_summary = all_metrics[all_metrics.role.eq("selection")].groupby(
    ["objective", "model"], as_index=False
).mean(numeric_only=True)
model_card = chr(10).join([
    "# CornerScout model card - notebook 05", "",
    f"- Run: `{RUN_ID}`", f"- Source: `{SOURCE_CONTRACT}` / `{contract04['run_id']}`",
    "- Validation: three expanding-origin development windows; final confirmation excluded from decisions.",
    "- Selection: probabilistic quality and discrimination improve in at least 2/3 windows without calibration degradation.",
    "- Bootstrap: 200 resamples of complete matches; never individual corners.",
    f"- Runtime: Python {platform.python_version()}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}.",
    "", "## Winners", winner_markdown,
    "", "## Development Metrics", "```text", metric_summary.to_string(index=False), "```",
    "", "## Limitations",
    *[f"- {item}" for item in limitations],
])
(OUT / "model_card.md").write_text(model_card, encoding="utf-8")
artifact_records.append({"file": "model_card.md", "role": "documentation",
                         "sha256": digest(OUT / "model_card.md"), "bytes": (OUT / "model_card.md").stat().st_size})

contract05 = {"stage": "05_modeling", "contract_version": STAGE_VERSION, "run_id": RUN_ID,
              "source_contract_version": SOURCE_CONTRACT, "source_run_id": contract04["run_id"],
              "windows": [{key: str(value.date()) if isinstance(value, pd.Timestamp) else value for key, value in row.items()} for row in WINDOWS],
              "final_period": {"start": str(FINAL_START.date()), "end_exclusive": str(FINAL_END.date()),
                               "used_for_decisions": False, "pristine_holdout": False,
                               "exception": contract04["post_eda_period_status"]},
              "selection_rule": "candidate improves probabilistic quality and discrimination in >=2/3 development windows without calibration degradation; otherwise reference wins",
              "automatic_balancing": False, "bootstrap_unit": "match", "bootstrap_iterations": BOOTSTRAPS,
              "kmeans_features_used": False, "winners": winner_rows, "limitations": limitations,
              "artifacts": artifact_records,
              "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                              "numpy": np.__version__, "scikit_learn": sklearn.__version__}}
write_json(OUT / "contract.json", contract05)
print(json.dumps({"winners": winner_rows, "final_period": contract05["final_period"]}, indent=2))

## Conclusion

Los ganadores anteriores proceden exclusivamente de datos reales y de tres
ventanas de desarrollo. La confirmacion final se informa pero no cambia ninguna
decision. Los artefactos `retrospective` son los usados para esa reevaluacion;
los `refit_full` incorporan toda la temporada y quedan explicitamente sin una
nueva estimacion fuera de muestra.